In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# TensorFlow and Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.utils import plot_model

# Additional utilities
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


In [ ]:
# Comprehensive LSTM vs GRU Comparison

class ModelComparison:
    """
    Compare LSTM and GRU models across multiple metrics
    """
    
    def __init__(self):
        self.results = {}
        
    def create_lstm_model(self, input_shape, units=64, layers=2, dropout=0.2):
        """Create LSTM model"""
        model = Sequential()
        
        # First LSTM layer
        model.add(LSTM(units, return_sequences=True, input_shape=input_shape))
        model.add(Dropout(dropout))
        
        # Additional LSTM layers
        for _ in range(layers - 1):
            model.add(LSTM(units, return_sequences=True))
            model.add(Dropout(dropout))
        
        # Final LSTM layer
        model.add(LSTM(units))
        model.add(Dropout(dropout))
        
        # Output layer
        model.add(Dense(1, activation='linear'))
        
        model.compile(optimizer=Adam(learning_rate=0.001), 
                     loss='mse', metrics=['mae'])
        return model
    
    def create_gru_model(self, input_shape, units=64, layers=2, dropout=0.2):
        """Create GRU model"""
        model = Sequential()
        
        # First GRU layer
        model.add(GRU(units, return_sequences=True, input_shape=input_shape))
        model.add(Dropout(dropout))
        
        # Additional GRU layers
        for _ in range(layers - 1):
            model.add(GRU(units, return_sequences=True))
            model.add(Dropout(dropout))
        
        # Final GRU layer
        model.add(GRU(units))
        model.add(Dropout(dropout))
        
        # Output layer
        model.add(Dense(1, activation='linear'))
        
        model.compile(optimizer=Adam(learning_rate=0.001), 
                     loss='mse', metrics=['mae'])
        return model
    
    def compare_models(self, X_train, y_train, X_test, y_test, epochs=50):
        """
        Compare LSTM and GRU models
        
        Returns:
            Dictionary with comparison results
        """
        input_shape = (X_train.shape[1], X_train.shape[2])
        
        # Create models
        lstm_model = self.create_lstm_model(input_shape)
        gru_model = self.create_gru_model(input_shape)
        
        print("LSTM Model Architecture:")
        lstm_model.summary()
        
        print("\\nGRU Model Architecture:")
        gru_model.summary()
        
        # Calculate model parameters
        lstm_params = lstm_model.count_params()
        gru_params = gru_model.count_params()
        
        print(f"\\nModel Parameters:")
        print(f"LSTM: {lstm_params:,}")
        print(f"GRU: {gru_params:,}")
        print(f"GRU has {((lstm_params - gru_params) / lstm_params * 100):.1f}% fewer parameters")
        
        # Training callbacks
        early_stopping = EarlyStopping(monitor='val_loss', patience=10, 
                                     restore_best_weights=True)
        
        # Train LSTM
        print("\\nTraining LSTM...")
        start_time = time.time()
        lstm_history = lstm_model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=epochs,
            batch_size=32,
            callbacks=[early_stopping],
            verbose=1
        )
        lstm_training_time = time.time() - start_time
        
        # Train GRU
        print("\\nTraining GRU...")
        start_time = time.time()
        gru_history = gru_model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=epochs,
            batch_size=32,
            callbacks=[early_stopping],
            verbose=1
        )
        gru_training_time = time.time() - start_time
        
        # Evaluate models
        lstm_test_loss = lstm_model.evaluate(X_test, y_test, verbose=0)
        gru_test_loss = gru_model.evaluate(X_test, y_test, verbose=0)
        
        # Make predictions
        lstm_predictions = lstm_model.predict(X_test, verbose=0)
        gru_predictions = gru_model.predict(X_test, verbose=0)
        
        # Calculate additional metrics
        lstm_mse = mean_squared_error(y_test, lstm_predictions)
        lstm_mae = mean_absolute_error(y_test, lstm_predictions)
        
        gru_mse = mean_squared_error(y_test, gru_predictions)
        gru_mae = mean_absolute_error(y_test, gru_predictions)
        
        # Store results
        self.results = {
            'lstm': {
                'model': lstm_model,
                'history': lstm_history,
                'test_loss': lstm_test_loss[0],
                'test_mae': lstm_test_loss[1],
                'mse': lstm_mse,
                'mae': lstm_mae,
                'rmse': np.sqrt(lstm_mse),
                'training_time': lstm_training_time,
                'parameters': lstm_params,
                'predictions': lstm_predictions
            },
            'gru': {
                'model': gru_model,
                'history': gru_history,
                'test_loss': gru_test_loss[0],
                'test_mae': gru_test_loss[1],
                'mse': gru_mse,
                'mae': gru_mae,
                'rmse': np.sqrt(gru_mse),
                'training_time': gru_training_time,
                'parameters': gru_params,
                'predictions': gru_predictions
            }
        }
        
        return self.results

# Initialize comparison class
comparison = ModelComparison()
print("Model comparison class initialized!")
